# ASSIGNMENT 5 NLP – Token Classification:Fine-Tuning DistilBERT for POS Tagging & Chunking
## Objective

To build and fine-tune a transformer model (BERT/DistilBERT) to perform Part-of-Speech (POS) Tagging and Chunking (Phrase Detection) using token classification techniques.


# Install the dependencies and import the required libraries

In [3]:
# Suppress warnings for clarity
import warnings
warnings.filterwarnings("ignore")

# Core libraries
import torch
import numpy as np
import pandas as pd

# Hugging Face ecosystem
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, Trainer, TrainingArguments
from transformers import DataCollatorForTokenClassification

# Evaluation
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

W0406 00:14:17.388000 19756 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


# Step 1: Dataset Selection

In [4]:
# Load CoNLL-2003 dataset for Chunking
dataset = load_dataset("conll2003")

# Inspect label categories
pos_labels   = dataset["train"].features["pos_tags"].feature.names
print("POS Labels:", pos_labels)
chunk_labels = dataset["train"].features["chunk_tags"].feature.names
print("Chunk Labels:", chunk_labels)

POS Labels: ['"', "''", '#', '$', '(', ')', ',', '.', ':', '``', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'LS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'NN|SYM', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB']
Chunk Labels: ['O', 'B-ADJP', 'I-ADJP', 'B-ADVP', 'I-ADVP', 'B-CONJP', 'I-CONJP', 'B-INTJ', 'I-INTJ', 'B-LST', 'I-LST', 'B-NP', 'I-NP', 'B-PP', 'I-PP', 'B-PRT', 'I-PRT', 'B-SBAR', 'I-SBAR', 'B-UCP', 'I-UCP', 'B-VP', 'I-VP']


# Step 2: Data Preprocessing

In [6]:
from transformers import AutoTokenizer

# Step 1: Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Step 2: Define preprocessing function
def tokenize_and_align_labels(examples, label_column):
    tokenized = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )
    labels = []

    for i in range(len(tokenized["input_ids"])):
        word_ids = tokenized.word_ids(batch_index=i)
        example_labels = examples[label_column][i]
        aligned_labels = []
        prev_word = None

        for word_id in word_ids:
            if word_id is None:
                aligned_labels.append(-100)
            elif word_id != prev_word:
                aligned_labels.append(example_labels[word_id])
            else:
                aligned_labels.append(-100)
            prev_word = word_id

        labels.append(aligned_labels)

    tokenized["labels"] = labels
    return tokenized

# Step 3: Apply preprocessing
tokenized_pos = dataset.map(lambda x: tokenize_and_align_labels(x, "pos_tags"), batched=True)
tokenized_chunk = dataset.map(lambda x: tokenize_and_align_labels(x, "chunk_tags"), batched=True)

# Step 4: Display samples
print("=== POS Sample ===")
print(tokenized_pos["train"][0])

print("\n=== Chunking Sample ===")
print(tokenized_chunk["train"][0])

# Pretty print tokens with labels for POS
sample = tokenized_pos["train"][0]
tokens = tokenizer.convert_ids_to_tokens(sample["input_ids"])
labels = sample["labels"]

print("\n=== Token-to-Label Alignment (POS Sample) ===")
for tok, lab in zip(tokens, labels):
    print(f"{tok:12} -> {lab}")


Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

=== POS Sample ===
{'id': '0', 'tokens': ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'], 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7], 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0], 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0], 'input_ids': [101, 7327, 19164, 2446, 2655, 2000, 17757, 2329, 12559, 1012, 102], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [-100, 22, 42, 16, 21, 35, 37, 16, 21, 7, -100]}

=== Chunking Sample ===
{'id': '0', 'tokens': ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'], 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7], 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0], 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0], 'input_ids': [101, 7327, 19164, 2446, 2655, 2000, 17757, 2329, 12559, 1012, 102], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [-100, 11, 21, 11, 12, 21, 22, 11, 12, 0, -100]}

=== Token-to-Label Alignment (POS Sample) ===
[CLS]        -> -100
eu           -> 22

# Step 3: Model Setup

In [7]:
# POS model
id2label_pos = {i: l for i, l in enumerate(pos_labels)}
label2id_pos = {l: i for i, l in enumerate(pos_labels)}

pos_model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(pos_labels),
    id2label=id2label_pos,
    label2id=label2id_pos
)

# Chunking model
id2label_chunk = {i: l for i, l in enumerate(chunk_labels)}
label2id_chunk = {l: i for i, l in enumerate(chunk_labels)}

chunk_model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(chunk_labels),
    id2label=id2label_chunk,
    label2id=label2id_chunk
)

Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# Step 4: Model Training

In [13]:
from transformers import TrainingArguments

# Define training arguments
training_args = TrainingArguments(
    output_dir="./results_pos_chunk",      
    evaluation_strategy="epoch",           
    save_strategy="epoch",                
    learning_rate=2e-5,                    
    per_device_train_batch_size=16,        
    per_device_eval_batch_size=16,
    num_train_epochs=3,                    
    weight_decay=0.01,                     
    logging_dir="./logs",                  
    logging_steps=50,                      
    report_to="none"                      
)

from transformers import DataCollatorForTokenClassification
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

import numpy as np
from datasets import load_metric

# Load seqeval metric (commonly used for token classification)
metric = load_metric("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (-100) when aligning labels
    true_predictions = [
        [str(pred) for (pred, lab) in zip(prediction, label) if lab != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [str(lab) for (pred, lab) in zip(prediction, label) if lab != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }



In [14]:
trainer_pos = Trainer(
    model=pos_model,
    args=training_args,
    train_dataset=tokenized_pos["train"],
    eval_dataset=tokenized_pos["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer_chunk = Trainer(
    model=chunk_model,
    args=training_args,
    train_dataset=tokenized_chunk["train"],
    eval_dataset=tokenized_chunk["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# Train separately
trainer_pos.train()
trainer_chunk.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.242000,0.256491,0.909975,0.911598,0.910786,0.937133
2,0.180100,0.229972,0.918748,0.915913,0.917329,0.942039
3,0.158700,0.219747,0.921209,0.920277,0.920743,0.944492


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.203700,0.201301,0.917873,0.917462,0.917667,0.949496
2,0.159000,0.177646,0.925319,0.924415,0.924867,0.953487
3,0.127100,0.171930,0.927502,0.925197,0.926348,0.954519


TrainOutput(global_step=2634, training_loss=0.2180498040980612, metrics={'train_runtime': 6866.7561, 'train_samples_per_second': 6.134, 'train_steps_per_second': 0.384, 'total_flos': 510251380802730.0, 'train_loss': 0.2180498040980612, 'epoch': 3.0})

# Step 5: Evaluation

In [15]:
pos_results   = trainer_pos.evaluate(tokenized_pos["test"])
chunk_results = trainer_chunk.evaluate(tokenized_chunk["test"])

print("POS Results:", pos_results)
print("Chunking Results:", chunk_results)

POS Results: {'eval_loss': 0.25867730379104614, 'eval_precision': 0.9115805946791862, 'eval_recall': 0.9082556035082835, 'eval_f1': 0.9099150615768701, 'eval_accuracy': 0.9393991601162915, 'eval_runtime': 111.4138, 'eval_samples_per_second': 30.993, 'eval_steps_per_second': 1.939, 'epoch': 3.0}
Chunking Results: {'eval_loss': 0.2031191885471344, 'eval_precision': 0.9198195622478478, 'eval_recall': 0.9149944699004582, 'eval_f1': 0.917400671693809, 'eval_accuracy': 0.9505330031226445, 'eval_runtime': 111.2464, 'eval_samples_per_second': 31.039, 'eval_steps_per_second': 1.942, 'epoch': 3.0}


In [16]:
sentence = "John works at Google in California"
tokens = sentence.split()
inputs = tokenizer(tokens, return_tensors="pt", is_split_into_words=True)

# POS predictions
outputs_pos = pos_model(**inputs)
pred_pos = torch.argmax(outputs_pos.logits, dim=-1).squeeze().tolist()
labels_pos = [id2label_pos[p] for p in pred_pos[:len(tokens)]]

# Chunk predictions
outputs_chunk = chunk_model(**inputs)
pred_chunk = torch.argmax(outputs_chunk.logits, dim=-1).squeeze().tolist()
labels_chunk = [id2label_chunk[p] for p in pred_chunk[:len(tokens)]]

# Display side by side
for word, pos, chunk in zip(tokens, labels_pos, labels_chunk):
    print(f"{word:12} | POS: {pos:8} | Chunk: {chunk}")

John         | POS: NN       | Chunk: B-NP
works        | POS: NNP      | Chunk: B-NP
at           | POS: VBZ      | Chunk: B-VP
Google       | POS: IN       | Chunk: B-PP
in           | POS: NNP      | Chunk: B-NP
California   | POS: IN       | Chunk: B-PP


## Task 7: Comparison 

| Aspect              | POS Tagging (Grammar-level) | Chunking (Phrase-level) |
|---------------------|-----------------------------|--------------------------|
| **Definition**      | Assigns grammatical categories (NOUN, VERB, ADJ, etc.) to each word | Groups words into syntactic phrases (NP, VP, PP, etc.) using BIO notation |
| **Granularity**     | Word-level labeling         | Span-level labeling (multi-word phrases) |
| **Difficulty**      | Easy – direct word-to-tag mapping | Medium – requires BIO span consistency |
| **Use Cases**       | Grammar analysis, lemmatization, parsing | Information extraction, shallow parsing, NER support |
| **Model Challenge** | Subword alignment but straightforward mapping | Span integrity, BIO scheme consistency, more complex evaluation |
| **Evaluation**      | Precision/Recall/F1 on single tags | Precision/Recall/F1 on phrase spans |

**Summary:**  
POS tagging is simpler because each word gets a single grammatical tag.  
Chunking is more complex since it involves grouping words into phrases, requiring BIO consistency and span-level evaluation.


## Task 8: Report 

### Differences Between POS Tagging and Chunking
- **POS Tagging**: Focuses on grammar categories (e.g., `NOUN`, `VERB`). It’s foundational for linguistic analysis and relatively easy for models to learn.  
- **Chunking**: Focuses on phrase boundaries (e.g., `B-NP`, `I-VP`). It’s more challenging because the model must maintain span consistency across multiple tokens.  

### Challenges Faced
- **Subword Tokenization**: BERT splits words into subwords, requiring careful label alignment (`-100` for subword continuations).  
- **BIO Scheme in Chunking**: Ensuring correct transitions between `B-` and `I-` tags is tricky.  
- **Evaluation Metrics**: Sequence-based metrics (seqeval) are stricter than token-level accuracy, so debugging errors requires careful inspection.  
- **Training Time**: Fine-tuning transformer models is computationally intensive, especially when training two parallel models (POS + Chunking).  

### Observations and Insights
- DistilBERT performs well for both tasks, but chunking F1 scores are usually lower than POS tagging due to span complexity.  
- POS tagging can serve as a “warm-up” task before tackling chunking.  
- Chunking provides richer linguistic information, useful for downstream tasks like information extraction and shallow parsing.  
- The same dataset (CoNLL‑2003) can be leveraged for both tasks, making it ideal for side-by-side comparison.  